# 01 Webcam-Based PM2.5 Dataset Pipeline

This notebook implements the complete workflow for constructing a webcam-based PM2.5 dataset, including:

- Webcam image loading and ROI selection
- Image feature extraction
- PM2.5, ERA5, and ARPA meteorological data processing
- Multi-source dataset merging and cleaning

The final output is an analysis-ready dataset for subsequent air quality modeling and analysis.


## Setup

In [1]:
import importlib
import json
import sys

from pathlib import Path

import pandas as pd
from IPython.display import display

## 1. Load Collected Webcam Images
Load all collected webcam images and initialize the project paths.

In [2]:
# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Define image directory
if IN_COLAB:
    drive.mount('/content/drive')

    # Google Drive folder
    IMAGE_DIR = Path('/content/drive/MyDrive/webcam_images')

else:
    # notebooks/ -> project root

    PROJECT_ROOT = Path.cwd().parent

    if str(PROJECT_ROOT) not in sys.path:

        sys.path.append(str(PROJECT_ROOT))

    IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "images"

image_files = list(IMAGE_DIR.glob("*.jpg"))

print("Running in Colab:", IN_COLAB)

if not IN_COLAB:
    print("Project root:", PROJECT_ROOT.name)

print(
    "Image directory:",
    IMAGE_DIR.relative_to(PROJECT_ROOT)
    if not IN_COLAB
    else IMAGE_DIR
)

print("Number of images:", len(image_files))

Running in Colab: False
Project root: webcam-pm25-toolbox
Image directory: data/raw/images
Number of images: 2799


## 2. ROI Extraction
Interactively select the region of interest (ROI) used for feature extraction.

In [3]:
import src.roi_viewer as roi_viewer

importlib.reload(roi_viewer)

roi_controls = roi_viewer.build_roi_viewer(IMAGE_DIR)

Output()

In [4]:
selected_roi = {
    "top": roi_controls["top"].value,
    "left": roi_controls["left"].value,
    "height": roi_controls["height"].value,
    "width": roi_controls["width"].value
}

ROI_JSON = PROJECT_ROOT / "config" / "roi.json"
ROI_JSON.parent.mkdir(parents=True, exist_ok=True)

with open(ROI_JSON, "w") as f:
    json.dump(selected_roi, f, indent=4)

print("Selected ROI:")
print(f"top = {selected_roi['top']}")
print(f"left = {selected_roi['left']}")
print(f"height = {selected_roi['height']}")
print(f"width = {selected_roi['width']}")
print(f"mode = {roi_controls['mode'].value}")

print("\nROI saved to:")
print(ROI_JSON.relative_to(PROJECT_ROOT))

Selected ROI:
top = 160
left = 116
height = 380
width = 515
mode = RGB

ROI saved to:
config/roi.json


## 3. Image Feature Extraction
Extract RGB, saturation, contrast, and B/R ratio features from all webcam images.

In [5]:
import src.image_features as image_features

importlib.reload(image_features)

IMAGE_FEATURES_CSV = PROJECT_ROOT / "data" / "interim" / "image_features.csv"

rows, skipped_files = image_features.extract_image_features(
    image_dir=PROJECT_ROOT / "data" / "raw" / "images",
    output_csv=IMAGE_FEATURES_CSV,
    roi=selected_roi
)

df = pd.read_csv(IMAGE_FEATURES_CSV)

df["image_path"] = df["image_path"].apply(
    lambda x: str(Path(x).relative_to(PROJECT_ROOT))
)

df.to_csv(IMAGE_FEATURES_CSV, index=False)

print(f"Image features extracted: {len(rows)} rows")
print("Saved CSV:", IMAGE_FEATURES_CSV.relative_to(PROJECT_ROOT))
print(f"Skipped files: {len(skipped_files)}")

df.head()

Image features extracted: 2799 rows
Saved CSV: data/interim/image_features.csv
Skipped files: 0


,datetime,R_roi,G_roi,B_roi,R_std,G_std,B_std,S_mean,V_mean,colorfulness,sky_brightness,contrast,gray_entropy,laplacian_variance,mean_gradient_magnitude,local_contrast,B_R_ratio,dark_pixel_ratio,sky_luminance_gradient,image_path
0,2026-03-03 03:00:00,84.341967,98.325697,119.317695,24.194092,23.540861,28.748329,0.299790,0.469556,23.377525,0.489513,23.899604,6.368111,2333.032878,8.740893,14.242407,1.414689,0.024921,10.621758,data/raw/images/20260303-0400.jpg
1,2026-03-03 04:00:00,84.496745,100.437798,121.108656,23.981469,23.530425,28.666245,0.308178,0.476231,23.219356,0.491991,23.866163,6.391382,2286.317946,8.571755,13.954696,1.433294,0.023526,10.900300,data/raw/images/20260303-0500.jpg
2,2026-03-03 05:00:00,86.156668,102.558431,123.832534,24.595152,24.272132,29.121707,0.310254,0.486596,23.177919,0.501450,24.562269,6.460171,2282.685316,8.488290,13.911710,1.437295,0.023275,11.379593,data/raw/images/20260303-0600.jpg
3,2026-03-03 06:00:00,107.791262,140.798753,171.635324,24.971811,29.975266,34.220437,0.376464,0.673097,29.415460,0.760342,28.779378,6.593807,931.629470,4.461193,7.771608,1.592293,0.009903,-14.525825,data/raw/images/20260303-0700.jpg
4,2026-03-03 07:00:00,117.449382,145.046152,168.683505,29.602077,31.820783,33.529277,0.312937,0.661505,23.877394,0.746854,31.179797,6.510708,1005.978398,4.422180,8.134474,1.436223,0.011390,0.362103,data/raw/images/20260303-0800.jpg


## 4. Download PM2.5 data
Download hourly PM2.5 observations from the EEA API and save them as a CSV file.

In [6]:
import src.eea_pm25_download as pm25

importlib.reload(pm25)

PM25_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "PM25_MI_hourly.csv"
)

PM25_TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pm25_temp"
)

pm25_df = pm25.download_pm25_data(
    api_start="2026-03-01T00:00:00Z",
    api_end="2026-07-21T23:00:00Z",
    station_prefix="IT/SPO.IT0477A_6001_BETA",
    temp_dir=PM25_TEMP_DIR,
    output_file=PM25_CSV,
    remove_temp=True
)

print(
    "Saved CSV:",
    PM25_CSV.relative_to(PROJECT_ROOT)
)
print(f"Rows: {len(pm25_df)}")

pm25_df.head()

Saved CSV: data/interim/PM25_MI_hourly.csv
Rows: 3402


,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,DataCapture,FkObservationLog
0,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 00:00:00,2026-03-01 01:00:00,36.062183000000000000,ug.m-3,hour,3,3,2026-03-01 01:00:00,None,9e664d80-8d0a-481e-82ca-112dac15e708
1,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 01:00:00,2026-03-01 02:00:00,25.287216000000000000,ug.m-3,hour,3,3,2026-03-01 02:00:00,None,1499e592-3c02-422b-8646-9dfc91309a4e
2,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 02:00:00,2026-03-01 03:00:00,26.743391000000000000,ug.m-3,hour,3,3,2026-03-01 03:00:00,None,ce0352c6-6450-4676-acd5-03581314f3f2
3,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 03:00:00,2026-03-01 04:00:00,30.829412000000000000,ug.m-3,hour,3,3,2026-03-01 04:00:00,None,d7edb6da-fdba-4913-b409-5fc4b3058c28
4,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 04:00:00,2026-03-01 05:00:00,33.409030000000000000,ug.m-3,hour,3,3,2026-03-01 05:00:00,None,c481d8a8-c512-4069-a081-8057f828c6ac


## 5. Download and Merge ERA5 data
Download ERA5 meteorological variables and merge single-level and pressure-level data.

In [9]:
import src.era5_download as era5

importlib.reload(era5)

ERA5_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "era5"

ERA5_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "era5_all_merged.csv"
)

# Run ERA5 download + processing
START_DATE = "2026-03-03"

END_DATE = "2026-07-20"

era5_df = era5.download_era5_data(

    lat=45.4642,

    lon=9.1900,

    start_date=START_DATE,

    end_date=END_DATE,

    work_dir=ERA5_RAW_DIR,

    output_file=ERA5_CSV,

    overwrite=False,)

print("ERA5 raw files saved to:",ERA5_RAW_DIR.relative_to(PROJECT_ROOT),)

print( "Merged ERA5 CSV saved to:",ERA5_CSV.relative_to(PROJECT_ROOT),)

display(era5_df.head())

display(era5_df.tail())

2026-07-22 12:50:32,613 INFO Request ID is c1c5e1aa-1206-423e-9418-8b77935f5b01
2026-07-22 12:50:32,711 INFO status has been updated to accepted
2026-07-22 12:50:56,724 INFO status has been updated to successful


4e81d3cf5569e440f5a27803220745f8.zip:   0%|          | 0.00/29.0k [00:00<?, ?B/s]

2026-07-22 12:50:58,632 INFO Request ID is 79deffbc-4e1e-41e6-bd5c-2669bcdda915
2026-07-22 12:50:58,711 INFO status has been updated to accepted
2026-07-22 12:51:20,779 INFO status has been updated to running
2026-07-22 12:55:20,012 INFO status has been updated to successful


7ce47981496693485422850f0da2be22.zip:   0%|          | 0.00/160k [00:00<?, ?B/s]

Completed ERA5 chunk: 20260303_20260331, 696 rows


...

2026-07-22 13:13:18,506 INFO Request ID is 8eab5298-e848-4dab-abb1-94e1d9ded23c
2026-07-22 13:13:18,596 INFO status has been updated to accepted
2026-07-22 13:13:41,075 INFO status has been updated to running
2026-07-22 13:17:39,621 INFO status has been updated to successful


3f16a59d61938fb31fc7e9b985d8045e.zip:   0%|          | 0.00/114k [00:00<?, ?B/s]

Completed ERA5 chunk: 20260701_20260720, 396 rows

ERA5 final quality check
------------------------
Expected rows: 3360
Actual rows: 3276
Missing timestamps: 84
Unexpected timestamps: 0
First missing timestamps:
[Timestamp('2026-07-17 12:00:00'), Timestamp('2026-07-17 13:00:00'), Timestamp('2026-07-17 14:00:00'), Timestamp('2026-07-17 15:00:00'), Timestamp('2026-07-17 16:00:00'), Timestamp('2026-07-17 17:00:00'), Timestamp('2026-07-17 18:00:00'), Timestamp('2026-07-17 19:00:00'), Timestamp('2026-07-17 20:00:00'), Timestamp('2026-07-17 21:00:00')]

Final ERA5 dataset saved
Path: /Users/qingxuan/Desktop/webcam-pm25-toolbox/data/interim/era5_all_merged.csv
Shape: (3276, 20)
Time range: 2026-03-03 00:00:00 to 2026-07-17 11:00:00
ERA5 raw files saved to: data/raw/era5
Merged ERA5 CSV saved to: data/interim/era5_all_merged.csv


,time,T2M,D2M,RH,U10,V10,SP,TP,BLH,TCC,CBH,WS10,GP_500,GP_850,T_500,T_850,U_500,U_850,V_500,V_850
0,2026-03-03 00:00:00,282.58673,280.93820,89.448424,-1.391312,-0.647079,101091.340,0.000015,68.070244,0.672577,390.71550,1.534425,55157.011719,15232.347656,249.124405,274.669922,11.939407,0.650497,-5.282013,1.839310
1,2026-03-03 01:00:00,282.36584,280.53723,88.338400,-1.382080,-0.660645,101102.170,0.000012,51.929825,0.658997,390.46832,1.531861,55172.214844,15234.148438,249.069839,274.547058,12.195877,1.366013,-4.574661,2.231323
2,2026-03-03 02:00:00,282.23172,280.21277,87.183727,-1.037506,-0.783157,101104.016,0.000011,47.110245,0.467834,406.95978,1.299906,55180.851562,15236.046875,248.951233,274.603455,12.443100,1.616684,-3.679062,1.787491
3,2026-03-03 03:00:00,282.31757,280.01376,85.506732,-1.028503,-0.998428,101104.090,0.000044,26.307983,0.403442,283.40190,1.433415,55171.746094,15233.457031,248.844803,274.421387,12.307693,1.781799,-3.885681,1.179291
4,2026-03-03 04:00:00,281.79294,279.64304,86.360749,-0.440567,-0.088943,101146.650,0.000115,16.255644,0.486603,356.12262,0.449455,55200.664062,15261.542969,248.774567,274.360657,11.759583,1.841400,-4.528091,0.527161


,time,T2M,D2M,RH,U10,V10,SP,TP,BLH,TCC,CBH,WS10,GP_500,GP_850,T_500,T_850,U_500,U_850,V_500,V_850
3271,2026-07-17 07:00:00,295.2270,289.24030,68.847014,0.765961,-2.205307,99958.780,0.000557,244.27087,0.744720,694.4486,2.334540,57406.722656,15020.664062,263.282715,289.576172,26.343445,-1.018768,4.656311,3.720428
3272,2026-07-17 08:00:00,297.7562,289.76940,61.122956,-0.376114,-2.366928,99942.890,0.000653,371.20310,0.508545,779.4624,2.396625,57420.289062,15033.027344,263.131104,289.733765,25.717407,0.428635,5.435989,3.460510
3273,2026-07-17 09:00:00,299.7678,289.82846,54.448054,-0.736984,-1.394470,99891.305,0.000376,489.69925,0.329407,1160.5963,1.577242,57406.500000,15020.839844,263.219025,289.852539,25.571381,1.056259,5.251877,2.397919
3274,2026-07-17 10:00:00,301.2848,294.68512,67.433252,1.327530,-0.901672,99840.440,0.000000,1068.34170,0.324921,978.5613,1.604789,57338.488281,14951.453125,263.977936,290.877441,28.816193,1.681641,4.239426,3.248444
3275,2026-07-17 11:00:00,302.4646,294.48676,62.212945,0.699890,0.978027,99806.170,0.000000,1109.83310,0.236053,1075.8596,1.202657,57374.570312,14957.773438,264.235657,291.761169,28.051407,2.505142,4.027740,2.547150


## 6. Merge ARPA station tables
ARPA meteorological station data were manually requested from ARPA Lombardia and provided as CSV tables.

This step merges the downloaded station tables into a unified hourly dataset.

In [11]:
import src.merge_arpa_data as arpa

importlib.reload(arpa)

ARPA_ROOT = PROJECT_ROOT / "data" / "raw" / "arpa"

ARPA_FILES = sorted(ARPA_ROOT.rglob("*.csv"))

ARPA_OUTPUT_FILE = PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv"

arpa_df = arpa.merge_arpa_tables(
    files=ARPA_FILES,
    output_file=ARPA_OUTPUT_FILE,
    time_column="Data-Ora",
    sensor_column="Id Sensore",
    utc_offset_hours=1,
    missing_value=-999,
)

print("ARPA merged CSV saved to:")
print(ARPA_OUTPUT_FILE.relative_to(PROJECT_ROOT))

display(arpa_df.head())

ARPA merged CSV saved to:
data/interim/arpa_merged.csv


,Data-Ora,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean
0,2025-12-31 23:00:00,3.3,248,81.3,<NA>
1,2026-01-01 00:00:00,2.9,300,83.0,0.3
2,2026-01-01 01:00:00,2.3,342,86.2,0.4
3,2026-01-01 02:00:00,1.9,4,87.4,<NA>
4,2026-01-01 03:00:00,1.4,20,89.4,0.4


## 7. Merge all datasets
This merges `image_features.csv`, `PM25_MI_hourly.csv`, `era5_all_merged.csv`, and `arpa_merged.csv` by UTC time.

In [7]:
import src.merge_all_data as merge_all

importlib.reload(merge_all)

MERGED_OUTPUT = PROJECT_ROOT / "data" / "interim" / "merged_dataset.csv"

merged_df = merge_all.merge_all_datasets(
    arpa_file=PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv",
    image_file=PROJECT_ROOT / "data" / "interim" / "image_features.csv",
    pm25_file=PROJECT_ROOT / "data" / "interim" / "PM25_MI_hourly.csv",
    era5_file=PROJECT_ROOT / "data" / "interim" / "era5_all_merged.csv",
    output_file=MERGED_OUTPUT,
)

print("Merged dataset saved to:")
print(MERGED_OUTPUT.relative_to(PROJECT_ROOT))

print(f"\nRows: {len(merged_df)}")
print(f"Columns: {merged_df.shape[1]}")

display(merged_df.head())

Merged dataset saved to:
data/interim/merged_dataset.csv

Rows: 2708
Columns: 45


,time,R_roi,G_roi,B_roi,R_std,G_std,B_std,S_mean,V_mean,colorfulness,...,T_500,T_850,U_500,U_850,V_500,V_850,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean
0,2026-03-03 03:00:00,84.341967,98.325697,119.317695,24.194092,23.540861,28.748329,0.299790,0.469556,23.377525,...,248.84480,274.42140,12.307693,1.781799,-3.885681,1.179291,10.5,346.0,89.8,NaN
1,2026-03-03 04:00:00,84.496745,100.437798,121.108656,23.981469,23.530425,28.666245,0.308178,0.476231,23.219356,...,248.77457,274.36066,11.759583,1.841400,-4.528091,0.527161,10.1,329.0,92.8,NaN
2,2026-03-03 05:00:00,86.156668,102.558431,123.832534,24.595152,24.272132,29.121707,0.310254,0.486596,23.177919,...,248.74742,274.40660,11.549728,1.765808,-4.992783,-0.205154,9.5,317.0,95.9,0.5
3,2026-03-03 06:00:00,107.791262,140.798753,171.635324,24.971811,29.975266,34.220437,0.376464,0.673097,29.415460,...,248.74470,274.49945,11.213989,1.240723,-5.396957,-0.769363,9.5,354.0,95.1,0.5
4,2026-03-03 07:00:00,117.449382,145.046152,168.683505,29.602077,31.820783,33.529277,0.312937,0.661505,23.877394,...,248.74140,274.53918,10.838104,0.719833,-5.423553,-0.914719,9.3,328.0,94.7,NaN
